In [1]:
# Cell 1: Import necessary libraries
import os
import shutil
import random
from pathlib import Path

# Set random seed for reproducibility
random.seed(42)

In [2]:
# Cell 2: Define your paths
# Update these paths to match your actual directory structure
base_path = "finaldata"  # Your main dataset folder
train_images_path = os.path.join(base_path, "train/images")
train_labels_path = os.path.join(base_path, "train/labels")
val_images_path = os.path.join(base_path, "valid/images")
val_labels_path = os.path.join(base_path, "valid/labels")
test_images_path = os.path.join(base_path, "test/images")
test_labels_path = os.path.join(base_path, "test/labels")
# Validation ratio (what percentage of training data to use for validation)
val_ratio = 0.2  # 20% for validation

print("Paths defined:")
print(f"Train images: {train_images_path}")
print(f"Train labels: {train_labels_path}")
print(f"Val images: {val_images_path}")
print(f"Val labels: {val_labels_path}")
print(f"Test images: {test_images_path}")
print(f"Test labels: {test_labels_path}")

Paths defined:
Train images: finaldata\train/images
Train labels: finaldata\train/labels
Val images: finaldata\valid/images
Val labels: finaldata\valid/labels
Test images: finaldata\test/images
Test labels: finaldata\test/labels


In [12]:
# Cell 3: Create validation directories
def create_val_directories():
    """Create the validation directory structure"""
    Path(val_images_path).mkdir(parents=True, exist_ok=True)
    Path(val_labels_path).mkdir(parents=True, exist_ok=True)
    print("✓ Created validation directories:")
    print(f"  - {val_images_path}")
    print(f"  - {val_labels_path}")

create_val_directories()

✓ Created validation directories:
  - Datasets/subset_dataset\valid/images
  - Datasets/subset_dataset\valid/labels


In [3]:
# Cell 4: Get all training image-label pairs
def get_training_pairs():
    """Get all image-label pairs from training directory"""
    image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}
    training_pairs = []
    
    # Get all image files from training directory
    for image_file in os.listdir(train_images_path):
        if any(image_file.lower().endswith(ext) for ext in image_extensions):
            image_name = Path(image_file).stem
            label_file = f"{image_name}.txt"
            label_path = os.path.join(train_labels_path, label_file)
            
            # Check if corresponding label exists
            if os.path.exists(label_path):
                training_pairs.append((image_file, label_file))
            else:
                print(f"Warning: No label found for {image_file}")
    
    print(f"Found {len(training_pairs)} training image-label pairs")
    return training_pairs

training_pairs = get_training_pairs()


Found 3626 training image-label pairs


In [10]:
# Cell 5: Split training data and move to validation
def create_validation_set(training_pairs, val_ratio=0.2):
    """Split training data and move portion to validation directory"""
    # Shuffle the pairs randomly
    random.shuffle(training_pairs)
    
    # Calculate how many samples to move to validation
    num_val = int(len(training_pairs) * val_ratio)
    val_pairs = training_pairs[:num_val]
    
    print(f"Moving {len(val_pairs)} samples to validation set")
    print(f"Keeping {len(training_pairs) - len(val_pairs)} samples in training set")
    
    # Move files to validation directory
    moved_count = 0
    for image_file, label_file in val_pairs:
        # Source paths
        src_image = os.path.join(train_images_path, image_file)
        src_label = os.path.join(train_labels_path, label_file)
        
        # Destination paths
        dst_image = os.path.join(val_images_path, image_file)
        dst_label = os.path.join(val_labels_path, label_file)
        
        # Move files
        shutil.move(src_image, dst_image)
        shutil.move(src_label, dst_label)
        moved_count += 1
    
    print(f"✓ Successfully moved {moved_count} image-label pairs to validation directory")

create_validation_set(training_pairs, val_ratio)

Moving 233 samples to validation set
Keeping 932 samples in training set
✓ Successfully moved 233 image-label pairs to validation directory


In [4]:
# Cell 6: Verify the results
def verify_split():
    """Verify that the split was successful"""
    # Count files in each directory
    train_images_count = len([f for f in os.listdir(train_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    train_labels_count = len([f for f in os.listdir(train_labels_path) if f.endswith('.txt')])
    val_images_count = len([f for f in os.listdir(val_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    val_labels_count = len([f for f in os.listdir(val_labels_path) if f.endswith('.txt')])
    test_images_count = len([f for f in os.listdir(test_images_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    test_labels_count = len([f for f in os.listdir(test_labels_path) if f.endswith('.txt')])

    print("Final counts:")
    print(f"Training images: {train_images_count}")
    print(f"Training labels: {train_labels_count}")
    print(f"Validation images: {val_images_count}")
    print(f"Validation labels: {val_labels_count}")
    print(f"Test images: {test_images_count}")
    print(f"Test labels: {test_labels_count}")
    print(f"Validation labels: {val_labels_count}")
    
    # Check if counts match
    if train_images_count == train_labels_count:
        print("✓ Training: All images have corresponding labels")
    else:
        print("✗ Training: Mismatch between images and labels")
    
    if val_images_count == val_labels_count:
        print("✓ Validation: All images have corresponding labels")
    else:
        print("✗ Validation: Mismatch between images and labels")

verify_split()

Final counts:
Training images: 3626
Training labels: 3626
Validation images: 1000
Validation labels: 1000
Test images: 501
Test labels: 501
Validation labels: 1000
✓ Training: All images have corresponding labels
✓ Validation: All images have corresponding labels


In [ ]:
# Cell 7: Final summary
print("""
VALIDATION SET CREATION COMPLETE!

What was accomplished:
✓ Created 'val/images' and 'val/labels' directories
✓ Randomly selected 20% of training data for validation
✓ Moved both images and corresponding labels to validation directory
✓ Maintained proper file mapping between images and labels

Your updated directory structure:
dataset/
├── train/
│   ├── images/     (remaining 80% of original training data)
│   └── labels/     (corresponding labels)
├── val/           ← NEW
│   ├── images/     (20% of original training data)
│   └── labels/     (corresponding labels)
└── test/
    ├── images/     (unchanged)
    └── labels/     (unchanged)

Next steps:
1. Update your YAML file to include: 'val: val/images'
2. Your data is now ready for YOLOv8 training!
""")

In [9]:
import os
import random
import shutil
import yaml
from pathlib import Path
from tqdm import tqdm

def create_subset_dataset(original_dataset_path, output_dataset_path, train_count=3000, val_count=1000, test_count=1000):
    """
    Create a subset of the original dataset with specified counts for train, val, and test sets.
    """
    
    # Define paths
    original_path = Path(original_dataset_path)
    output_path = Path(output_dataset_path)
    
    # First, check the dataset structure
    print(f"\nChecking dataset structure in: {original_path}")
    print("Contents of dataset directory:")
    
    # List all directories and files
    items = list(original_path.iterdir())
    if not items:
        print("ERROR: Dataset directory is empty!")
        return
    
    for item in items:
        if item.is_dir():
            # List subdirectories if they exist
            subitems = list(item.iterdir())
            subinfo = f" ({len(subitems)} items)" if len(subitems) <= 10 else f" ({len(subitems)} items...)"
            print(f"  - {item.name}/{subinfo}")
        else:
            print(f"  - {item.name}")
    
    # Look for possible dataset structures
    found_splits = {}
    
    # Check for standard YOLO structure
    standard_splits = ['train', 'valid', 'validation', 'val', 'test']
    for split in standard_splits:
        split_path = original_path / split
        if split_path.exists() and split_path.is_dir():
            found_splits[split] = split_path
    
    # If no standard splits found, check for images and labels directories
    if not found_splits:
        print("\nNo standard train/val/test directories found.")
        print("Looking for images/ and labels/ directories...")
        
        images_dir = original_path / 'images'
        labels_dir = original_path / 'labels'
        
        if images_dir.exists() and labels_dir.exists():
            print("Found images/ and labels/ directories.")
            return create_dataset_from_single_split(
                original_path, output_path, images_dir, labels_dir,
                train_count, val_count, test_count
            )
        else:
            print("ERROR: Could not find standard dataset structure.")
            print("\nExpected structure 1 (YOLO format):")
            print("  dataset/")
            print("  ├── train/")
            print("  │   ├── images/")
            print("  │   └── labels/")
            print("  ├── valid/")
            print("  │   ├── images/")
            print("  │   └── labels/")
            print("  ├── test/")
            print("  │   ├── images/")
            print("  │   └── labels/")
            print("  └── data.yaml")
            
            print("\nExpected structure 2 (Single split):")
            print("  dataset/")
            print("  ├── images/")
            print("  ├── labels/")
            print("  └── data.yaml")
            return
    
    print(f"\nFound the following splits: {list(found_splits.keys())}")
    
    # Map found splits to standard names
    split_mapping = {}
    if 'train' in found_splits:
        split_mapping['train'] = 'train'
    elif 'training' in found_splits:
        split_mapping['train'] = 'training'
    
    if 'valid' in found_splits:
        split_mapping['valid'] = 'valid'
    elif 'validation' in found_splits:
        split_mapping['valid'] = 'validation'
    elif 'val' in found_splits:
        split_mapping['valid'] = 'val'
    
    if 'test' in found_splits:
        split_mapping['test'] = 'test'
    elif 'testing' in found_splits:
        split_mapping['test'] = 'testing'
    
    # Create output directories
    output_dirs = {}
    for split in ['train', 'valid', 'test']:
        output_dirs[split] = output_path / split
        (output_dirs[split] / 'images').mkdir(parents=True, exist_ok=True)
        (output_dirs[split] / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Read YAML file if it exists
    yaml_path = original_path / 'data.yaml'
    if yaml_path.exists():
        with open(yaml_path, 'r') as f:
            yaml_data = yaml.safe_load(f)
            print(f"\nLoaded data.yaml with {len(yaml_data.get('names', []))} classes")
    
    # Collect all images from each split
    all_images = {'train': [], 'valid': [], 'test': []}
    
    for target_split in ['train', 'valid', 'test']:
        if target_split not in split_mapping:
            print(f"\n{target_split.capitalize()} split not found in dataset")
            continue
        
        original_split_name = split_mapping[target_split]
        split_path = found_splits[original_split_name]
        
        print(f"\nProcessing {original_split_name} as {target_split}...")
        
        # Try different possible structures
        possible_image_dirs = [
            split_path / 'images',
            split_path
        ]
        
        possible_label_dirs = [
            split_path / 'labels',
            split_path
        ]
        
        image_dir = None
        label_dir = None
        
        # Find images directory
        for dir_path in possible_image_dirs:
            if dir_path.exists():
                image_dir = dir_path
                break
        
        # Find labels directory
        for dir_path in possible_label_dirs:
            if dir_path.exists():
                label_dir = dir_path
                break
        
        if not image_dir:
            print(f"  Warning: Could not find images in {split_path}")
            continue
        
        print(f"  Looking for images in: {image_dir.relative_to(original_path)}")
        print(f"  Looking for labels in: {label_dir.relative_to(original_path) if label_dir else 'Not found'}")
        
        # Find all image files
        image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.JPG', '.JPEG', '.PNG']
        image_files = []
        
        for ext in image_extensions:
            image_files.extend(list(image_dir.glob(f'*{ext}')))
            image_files.extend(list(image_dir.glob(f'*{ext.upper()}')))
        
        print(f"  Found {len(image_files)} image files")
        
        # Create image-label pairs
        image_label_pairs = []
        missing_labels = 0
        
        for img_path in image_files:
            # Try to find corresponding label
            label_found = False
            
            if label_dir:
                # Try with .txt extension
                label_path = label_dir / f"{img_path.stem}.txt"
                if label_path.exists():
                    image_label_pairs.append((img_path, label_path))
                    label_found = True
                else:
                    # Try with different extensions
                    for ext in ['.txt', '.TXT']:
                        label_path = label_dir / f"{img_path.stem}{ext}"
                        if label_path.exists():
                            image_label_pairs.append((img_path, label_path))
                            label_found = True
                            break
            
            if not label_found:
                missing_labels += 1
                # print(f"    Warning: No label found for {img_path.name}")
        
        all_images[target_split] = image_label_pairs
        print(f"  Created {len(image_label_pairs)} image-label pairs")
        if missing_labels > 0:
            print(f"  Warning: {missing_labels} images without labels")
    
    # Calculate available counts
    available_counts = {
        'train': len(all_images['train']),
        'valid': len(all_images['valid']),
        'test': len(all_images['test'])
    }
    
    # Calculate required counts
    required_counts = {
        'train': min(train_count, available_counts['train']),
        'valid': min(val_count, available_counts['valid']),
        'test': min(test_count, available_counts['test'])
    }
    
    print(f"\nDataset Summary:")
    total_available = sum(available_counts.values())
    total_requested = train_count + val_count + test_count
    
    for split in ['train', 'valid', 'test']:
        print(f"  {split.capitalize()}: {available_counts[split]} available, {required_counts[split]} will be used")
    
    print(f"\n  Total available: {total_available}")
    print(f"  Total requested: {total_requested}")
    
    if total_available == 0:
        print("\nERROR: No images found in the dataset!")
        return
    
    if total_requested > total_available:
        print(f"\nWarning: Requested {total_requested} images but only {total_available} available")
    
    # Randomly sample images for each split
    for split in ['train', 'valid', 'test']:
        if required_counts[split] == 0:
            print(f"\nSkipping {split} (0 images requested)")
            continue
        
        if not all_images[split]:
            print(f"\nWarning: No images available for {split}")
            continue
        
        # Shuffle and select
        random.shuffle(all_images[split])
        selected_pairs = all_images[split][:required_counts[split]]
        
        print(f"\nCopying {len(selected_pairs)} files for {split}...")
        
        # Copy files
        for img_path, label_path in tqdm(selected_pairs, desc=f"{split}"):
            # Copy image
            dest_img_path = output_dirs[split] / 'images' / img_path.name
            shutil.copy2(img_path, dest_img_path)
            
            # Copy label
            dest_label_path = output_dirs[split] / 'labels' / label_path.name
            shutil.copy2(label_path, dest_label_path)
    
    # Create new YAML file
    if yaml_path.exists():
        new_yaml_data = yaml_data.copy()
        
        # Update paths
        new_yaml_data['path'] = str(output_path.resolve())
        new_yaml_data['train'] = 'train/images' if required_counts['train'] > 0 else ''
        new_yaml_data['val'] = 'valid/images' if required_counts['valid'] > 0 else ''
        new_yaml_data['test'] = 'test/images' if required_counts['test'] > 0 else ''
        
        # Save new YAML
        new_yaml_path = output_path / 'data.yaml'
        with open(new_yaml_path, 'w') as f:
            yaml.dump(new_yaml_data, f, default_flow_style=False)
        
        print(f"\nCreated new data.yaml at: {new_yaml_path}")
    else:
        # Create a basic YAML file
        new_yaml_data = {
            'path': str(output_path.resolve()),
            'train': 'train/images',
            'val': 'valid/images',
            'test': 'test/images',
            'names': ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']  # Update with your classes
        }
        
        new_yaml_path = output_path / 'data.yaml'
        with open(new_yaml_path, 'w') as f:
            yaml.dump(new_yaml_data, f, default_flow_style=False)
        
        print(f"\nCreated basic data.yaml at: {new_yaml_path}")
        print("Warning: Please update the 'names' field with your actual class names")
    
    # Print final summary
    print(f"\nSubset dataset created successfully at: {output_path}")
    print("\nFinal dataset structure:")
    for split in ['train', 'valid', 'test']:
        split_path = output_dirs[split]
        images = list((split_path / 'images').glob('*'))
        labels = list((split_path / 'labels').glob('*'))
        print(f"  {split}/:")
        print(f"    images/: {len(images)} files")
        print(f"    labels/: {len(labels)} files")

def create_dataset_from_single_split(original_path, output_path, images_dir, labels_dir, train_count, val_count, test_count):
    """Create dataset from a single split (all images in one directory)"""
    print("\nProcessing single-split dataset...")
    
    # Find all images
    image_extensions = ['.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.JPG', '.JPEG', '.PNG']
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(list(images_dir.glob(f'*{ext}')))
    
    print(f"Found {len(image_files)} images in total")
    
    # Create image-label pairs
    image_label_pairs = []
    missing_labels = 0
    
    for img_path in image_files:
        label_path = labels_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            image_label_pairs.append((img_path, label_path))
        else:
            missing_labels += 1
    
    print(f"Created {len(image_label_pairs)} image-label pairs")
    if missing_labels > 0:
        print(f"Warning: {missing_labels} images without labels")
    
    if len(image_label_pairs) == 0:
        print("ERROR: No valid image-label pairs found!")
        return
    
    # Shuffle all pairs
    random.shuffle(image_label_pairs)
    
    # Adjust counts if needed
    total_available = len(image_label_pairs)
    total_requested = train_count + val_count + test_count
    
    if total_requested > total_available:
        print(f"Adjusting counts: requested {total_requested}, available {total_available}")
        train_count = int(total_available * 0.6)
        val_count = int(total_available * 0.2)
        test_count = total_available - train_count - val_count
    
    # Split the data
    train_end = train_count
    val_end = train_end + val_count
    test_end = val_end + test_count
    
    train_pairs = image_label_pairs[:train_end]
    val_pairs = image_label_pairs[train_end:val_end]
    test_pairs = image_label_pairs[val_end:test_end]
    
    # Create output directories
    output_dirs = {}
    for split in ['train', 'valid', 'test']:
        output_dirs[split] = output_path / split
        (output_dirs[split] / 'images').mkdir(parents=True, exist_ok=True)
        (output_dirs[split] / 'labels').mkdir(parents=True, exist_ok=True)
    
    # Copy files
    splits = {
        'train': train_pairs,
        'valid': val_pairs,
        'test': test_pairs
    }
    
    for split, pairs in splits.items():
        if pairs:
            print(f"\nCopying {len(pairs)} files for {split}...")
            for img_path, label_path in tqdm(pairs, desc=split):
                dest_img = output_dirs[split] / 'images' / img_path.name
                dest_label = output_dirs[split] / 'labels' / label_path.name
                shutil.copy2(img_path, dest_img)
                shutil.copy2(label_path, dest_label)
    
    # Create YAML file
    yaml_path = original_path / 'data.yaml'
    if yaml_path.exists():
        with open(yaml_path, 'r') as f:
            yaml_data = yaml.safe_load(f)
    else:
        yaml_data = {'names': ['class_0', 'class_1', 'class_2', 'class_3', 'class_4']}
    
    new_yaml_data = yaml_data.copy()
    new_yaml_data['path'] = str(output_path.resolve())
    new_yaml_data['train'] = 'train/images'
    new_yaml_data['val'] = 'valid/images'
    new_yaml_data['test'] = 'test/images'
    
    new_yaml_path = output_path / 'data.yaml'
    with open(new_yaml_path, 'w') as f:
        yaml.dump(new_yaml_data, f, default_flow_style=False)
    
    print(f"\nDataset created successfully!")
    print(f"  Train: {len(train_pairs)} images")
    print(f"  Valid: {len(val_pairs)} images")
    print(f"  Test: {len(test_pairs)} images")

def create_balanced_subset(original_dataset_path, output_dataset_path, total_count=5000, train_ratio=0.6, val_ratio=0.2):
    """
    Create a balanced subset with automatic split calculations.
    """
    train_count = int(total_count * train_ratio)
    val_count = int(total_count * val_ratio)
    test_count = total_count - train_count - val_count
    
    print(f"Creating balanced subset:")
    print(f"  Total images: {total_count}")
    print(f"  Train: {train_count} ({train_ratio*100:.0f}%)")
    print(f"  Validation: {val_count} ({val_ratio*100:.0f}%)")
    print(f"  Test: {test_count} ({(1-train_ratio-val_ratio)*100:.0f}%)")
    
    create_subset_dataset(
        original_dataset_path=original_dataset_path,
        output_dataset_path=output_dataset_path,
        train_count=train_count,
        val_count=val_count,
        test_count=test_count
    )

if __name__ == "__main__":
    # ============================================
    # CONFIGURATION
    # ============================================
    
    # Path to your original dataset
    ORIGINAL_DATASET_PATH = "Datasets/underwater_dataset_yolov5.v1i.yolov8"
    
    # Path where the new subset will be created
    OUTPUT_DATASET_PATH = "Datasets/subset_dataset"
    
    # Option 1: Use exact counts
    USE_EXACT_COUNTS = False  # Set to True to use exact counts
    TRAIN_COUNT = 3000
    VAL_COUNT = 1000
    TEST_COUNT = 1000
    
    # Option 2: Use balanced subset with ratios
    TOTAL_COUNT = 5000
    TRAIN_RATIO = 0.6  # 60% for training
    VAL_RATIO = 0.2    # 20% for validation
    # Remaining 20% will be for test
    
    # ============================================
    # EXECUTION
    # ============================================
    
    # First, let's check if the original dataset exists
    if not os.path.exists(ORIGINAL_DATASET_PATH):
        print(f"ERROR: Dataset path does not exist: {ORIGINAL_DATASET_PATH}")
        print("Please check the path and try again.")
    else:
        if USE_EXACT_COUNTS:
            create_subset_dataset(
                original_dataset_path=ORIGINAL_DATASET_PATH,
                output_dataset_path=OUTPUT_DATASET_PATH,
                train_count=TRAIN_COUNT,
                val_count=VAL_COUNT,
                test_count=TEST_COUNT
            )
        else:
            create_balanced_subset(
                original_dataset_path=ORIGINAL_DATASET_PATH,
                output_dataset_path=OUTPUT_DATASET_PATH,
                total_count=TOTAL_COUNT,
                train_ratio=TRAIN_RATIO,
                val_ratio=VAL_RATIO
            )

Creating balanced subset:
  Total images: 5000
  Train: 3000 (60%)
  Validation: 1000 (20%)
  Test: 1000 (20%)

Checking dataset structure in: Datasets\underwater_dataset_yolov5.v1i.yolov8
Contents of dataset directory:
  - data.yaml
  - README.dataset.txt
  - README.roboflow.txt
  - test/ (2 items)
  - train/ (2 items)
  - valid/ (2 items)

Found the following splits: ['train', 'valid', 'test']

Loaded data.yaml with 5 classes

Processing train as train...
  Looking for images in: train\images
  Looking for labels in: train\labels
  Found 72160 image files
  Created 72160 image-label pairs

Processing valid as valid...
  Looking for images in: valid\images
  Looking for labels in: valid\labels
  Found 10164 image files
  Created 10164 image-label pairs

Processing test as test...
  Looking for images in: test\images
  Looking for labels in: test\labels
  Found 5088 image files
  Created 5088 image-label pairs

Dataset Summary:
  Train: 72160 available, 3000 will be used
  Valid: 10164

train: 100%|██████████| 3000/3000 [00:13<00:00, 219.94it/s]



Copying 1000 files for valid...


valid: 100%|██████████| 1000/1000 [00:03<00:00, 250.69it/s]



Copying 1000 files for test...


test: 100%|██████████| 1000/1000 [00:03<00:00, 277.98it/s]



Created new data.yaml at: Datasets\subset_dataset\data.yaml

Subset dataset created successfully at: Datasets\subset_dataset

Final dataset structure:
  train/:
    images/: 2823 files
    labels/: 2823 files
  valid/:
    images/: 859 files
    labels/: 859 files
  test/:
    images/: 741 files
    labels/: 741 files
